Google Colab Project Setup

Check GPU

In [ ]:
import sys
import os
import platform

print("Python version :", sys.version)
print("Platform       :", platform.platform())

# Check GPU
!nvidia-smi

Python version : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform       : Linux-6.6.122+-x86_64-with-glibc2.39
/bin/bash: line 1: nvidia-smi: command not found


Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Create our project directories

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/GC210_Retail_Forecasting"

directories = [
    PROJECT_DIR,
    f"{PROJECT_DIR}/data",
    f"{PROJECT_DIR}/data/raw",
    f"{PROJECT_DIR}/data/processed",
    f"{PROJECT_DIR}/models",
    f"{PROJECT_DIR}/results",
    f"{PROJECT_DIR}/results/metrics",
    f"{PROJECT_DIR}/results/predictions",
    f"{PROJECT_DIR}/results/figures",
    f"{PROJECT_DIR}/notebooks"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("Project directory created:")
print(PROJECT_DIR)

Project directory created:
/content/drive/MyDrive/GC210_Retail_Forecasting


Install required packages

In [ ]:
!pip install -q kaggle pandas numpy scikit-learn matplotlib seaborn tensorflow xgboost lightgbm shap optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 22.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import sklearn
import tensorflow as tf
import xgboost
import lightgbm
import shap
import optuna

print("Pandas       :", pd.__version__)
print("NumPy        :", np.__version__)
print("Scikit-learn :", sklearn.__version__)
print("TensorFlow   :", tf.__version__)
print("XGBoost      :", xgboost.__version__)
print("LightGBM     :", lightgbm.__version__)
print("SHAP         :", shap.__version__)
print("Optuna       :", optuna.__version__)

Pandas       : 2.2.3
NumPy        : 2.1.3
Scikit-learn : 1.6.1
TensorFlow   : 2.20.0
XGBoost      : 3.4.1
LightGBM     : 4.6.0
SHAP         : 0.52.0
Optuna       : 5.0.0


Load the three important files

In [ ]:
sales_path = f"/content/drive/MyDrive/DATASET/sales_train_evaluation.csv"
calendar_path = f"/content/drive/MyDrive/DATASET/calendar.csv"
prices_path = f"/content/drive/MyDrive/DATASET/sell_prices.csv"

sales = pd.read_csv(sales_path)
calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)

print("Sales shape    :", sales.shape)
print("Calendar shape :", calendar.shape)
print("Prices shape   :", prices.shape)

Sales shape    : (30490, 1947)
Calendar shape : (1969, 14)
Prices shape   : (6841121, 4)


Inspect the sales dataset

In [ ]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), object(6)
memory usage: 452.9+ MB


In [ ]:
sales.columns[:20]

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1',
       'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11',
       'd_12', 'd_13', 'd_14'],
      dtype='object')

Inspect calendar

In [ ]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [ ]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          1969 non-null   object
 1   wm_yr_wk      1969 non-null   int64 
 2   weekday       1969 non-null   object
 3   wday          1969 non-null   int64 
 4   month         1969 non-null   int64 
 5   year          1969 non-null   int64 
 6   d             1969 non-null   object
 7   event_name_1  162 non-null    object
 8   event_type_1  162 non-null    object
 9   event_name_2  5 non-null      object
 10  event_type_2  5 non-null      object
 11  snap_CA       1969 non-null   int64 
 12  snap_TX       1969 non-null   int64 
 13  snap_WI       1969 non-null   int64 
dtypes: int64(7), object(7)
memory usage: 215.5+ KB


In [ ]:
calendar.columns.tolist()

['date',
 'wm_yr_wk',
 'weekday',
 'wday',
 'month',
 'year',
 'd',
 'event_name_1',
 'event_type_1',
 'event_name_2',
 'event_type_2',
 'snap_CA',
 'snap_TX',
 'snap_WI']

In [ ]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [ ]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    object 
 1   item_id     object 
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 208.8+ MB


In [ ]:
prices.columns.tolist()

['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Basic data-quality checks

In [ ]:
print("Missing values in sales:")
print(sales.isna().sum().sum())

print("\nMissing values in calendar:")
print(calendar.isna().sum().sum())

print("\nMissing values in prices:")
print(prices.isna().sum().sum())

Missing values in sales:
0

Missing values in calendar:
7542

Missing values in prices:
0


Save a project configuration

In [ ]:
import json

RAW_DIR = f"{PROJECT_DIR}/data/raw"

config = {
    "project_name": "GC210_Retail_Forecasting",
    "dataset": "M5 Forecasting - Accuracy",
    "project_dir": PROJECT_DIR,
    "raw_data_dir": RAW_DIR,
    "random_seed": 42,
    "forecast_horizon": 28
}

config_path = f"{PROJECT_DIR}/config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved to:")
print(config_path)

Configuration saved to:
/content/drive/MyDrive/GC210_Retail_Forecasting/config.json


Set random seeds

In [ ]:
import random
import numpy as np
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


Final setup check

In [ ]:
print("=" * 60)
print("PROJECT SETUP COMPLETE")
print("=" * 60)

print(f"Project directory : {PROJECT_DIR}")
print(f"Raw data          : {RAW_DIR}")
print()

print("Dataset:")
print(f"  Sales            : {sales.shape}")
print(f"  Calendar         : {calendar.shape}")
print(f"  Prices           : {prices.shape}")

print()
print("Forecast horizon  : 28 days")
print("Random seed       : 42")
print("=" * 60)

PROJECT SETUP COMPLETE
Project directory : /content/drive/MyDrive/GC210_Retail_Forecasting
Raw data          : /content/drive/MyDrive/GC210_Retail_Forecasting/data/raw

Dataset:
  Sales            : (30490, 1947)
  Calendar         : (1969, 14)
  Prices           : (6841121, 4)

Forecast horizon  : 28 days
Random seed       : 42
